# Generating type 4 clones with LLMs

## Generation

In [1]:
DATASET_PATH = "../dataset/bigcodebench_normalized.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json"
OLLAMA_MODEL = "llama3.1:latest"   

# Generation settings
LLM_OPTS = {
    "temperature": 0.6,
    "top_p": 0.95,
    "repeat_penalty": 1.05,
    "num_predict": 768,   
}

N_ENTRIES = 4
CLONES_PER_ENTRY = 2  

In [2]:
import os, json  
from src.clone_gen import STRATEGY_HINTS, SYSTEM_PROMPT, build_user_prompt, generate_clones

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

sample = data[:N_ENTRIES]

results = []
for i, entry in enumerate(sample, 1):
    print(f"\nGenerating clones for Entry {i}/{len(sample)} | id={entry['id']}")
    clones = []

    original_body = entry["original_code"]
    tests_list    = entry["test"]
    description   = entry.get("description", "")
    libs          = entry.get("metadata", {}).get("libs", [])

    tests_snippet = tests_list[0] if tests_list else ""

    for k in range(CLONES_PER_ENTRY):
        hint = STRATEGY_HINTS[k % len(STRATEGY_HINTS)]
        user_prompt = build_user_prompt(original_body, description, libs, tests_snippet, hint)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ]

        try:
            code = generate_clones(messages, model=OLLAMA_MODEL, options=LLM_OPTS, expected_func_name="task_func")
            clones.append({
                "transformation": f"LLM/{OLLAMA_MODEL}",
                "strategy_hint": hint,
                "code": code
            })
        except Exception as e:
            print(f" Error generating clone {k+1}: {e}")

    results.append({
        "id": entry["id"],
        "language": entry["language"],
        "description": description,
        "metadata": entry.get("metadata", {}),
        "original_code": original_body,
        "test": tests_list,
        "clones": clones
    })

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)



Generating clones for Entry 1/4 | id=BigCodeBench/0

Generating clones for Entry 2/4 | id=BigCodeBench/1

Generating clones for Entry 3/4 | id=BigCodeBench/2

Generating clones for Entry 4/4 | id=BigCodeBench/3


## Running the tests on the clones

In [3]:
import os, json
from src.utils import validate_with_unittest

with open(OUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

results = []
for i, entry in enumerate(data, 1):
    print(f"\nTesting Entry {i}/{len(data)} | id={entry['id']}")
    clones = []

    tests_list = entry["test"]

    for k, clone in enumerate(entry.get("clones", []),1):
        try:
            code = clone["code"]
            # Get individual test results
            test_results = validate_with_unittest(code, tests_list) # this function is defined in 2.preprocess.ipynb
            clone["test_results"] = test_results

            # Print summary
            passed = sum(1 for v in test_results.values() if v=="PASS")
            total = len(test_results)
            print(f"  Clone {k}: {passed}/{total} tests passed")

            clones.append(clone)
        except Exception as e:
            print(f"  Error testing clone {k}: {e}")
            clone["test_results"] = {}
            clones.append(clone)

    entry["clones"] = clones
    results.append(entry)

# Save dataset with test results
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Done. Saved dataset with test results to {OUT_PATH}")



Testing Entry 1/4 | id=BigCodeBench/0
  Clone 1: 10/10 tests passed
  Clone 2: 10/10 tests passed

Testing Entry 2/4 | id=BigCodeBench/1
  Clone 1: 3/3 tests passed
  Clone 2: 3/3 tests passed

Testing Entry 3/4 | id=BigCodeBench/2
  Clone 1: 3/5 tests passed
  Clone 2: 5/5 tests passed

Testing Entry 4/4 | id=BigCodeBench/3
  Clone 1: 5/5 tests passed
  Clone 2: 5/5 tests passed

✅ Done. Saved dataset with test results to ../results/bigcodebench_llm_clones.json


## Checking clone type

In [4]:
import json
from codebleu import calc_codebleu  

# === Load clones dataset ===
with open(OUT_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# === Compute CodeBLEU for each clone ===
for idx, entry in enumerate(dataset):
    original_code = entry["original_code"]
    clones = entry.get("clones", [])
    for clone in clones:
        try:
            clone_code = clone["code"]
            # CodeBLEU expects lists: (refs, hyps, lang)
            score_dict = calc_codebleu([original_code], [clone_code], lang="python", weights=(0.25, 0.25, 0.25, 0.25), tokenizer=None)
            codebleu_score = score_dict["codebleu"]
            
            # Store in results
            clone["metrics"] = {"codebleu": codebleu_score}
            print(f"[Entry {idx}] Clone by {clone['transformation']} - CodeBLEU: {codebleu_score:.4f}")
        
        except Exception as e:
            print(f"Error scoring clone for entry {idx}: {e}")
            clone["metrics"] = {"codebleu": None}

# === Save dataset with metrics ===
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print("\n✅ Finished scoring all clones with CodeBLEU.")


[Entry 0] Clone by LLM/llama3.1:latest - CodeBLEU: 0.6427
[Entry 0] Clone by LLM/llama3.1:latest - CodeBLEU: 0.4327
[Entry 1] Clone by LLM/llama3.1:latest - CodeBLEU: 0.6659
[Entry 1] Clone by LLM/llama3.1:latest - CodeBLEU: 0.8088
[Entry 2] Clone by LLM/llama3.1:latest - CodeBLEU: 0.5358
[Entry 2] Clone by LLM/llama3.1:latest - CodeBLEU: 0.9462
[Entry 3] Clone by LLM/llama3.1:latest - CodeBLEU: 0.5729
[Entry 3] Clone by LLM/llama3.1:latest - CodeBLEU: 0.9532

✅ Finished scoring all clones with CodeBLEU.
